### Bibliotecas e Frameworks

In [ ]:
import os

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["KERAS_BACKEND"] = "tensorflow"

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

import tensorflow as tf
import keras
import keras.layers as lay
import matplotlib.pyplot as plt


2025-09-01 16:54:37.099552: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-01 16:54:37.160309: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-01 16:54:38.734625: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Questão 3

Implemente e treine uma Rede Neural Convolucional (CNN) para resolver o problema de classificação de objetos em imagens, utilizando a base de dados CIFAR-10, disponível em:

https://www.cs.toronto.edu/~kriz/cifar.html
 
Apresente a curva do erro médio ao longo do treinamento, bem como a matriz de confusão do modelo avaliado sobre o conjunto de testes. 

In [2]:
batch_size = 2**4

(trainx, trainy), (valx, valy) = keras.datasets.cifar10.load_data()

trainx = (trainx.astype("float32") - 127.5) / 127.5
valx = (valx.astype("float32") - 127.5) / 127.5

train = tf.data.Dataset.from_tensor_slices((trainx, trainy))
train = train.cache()
train = train.shuffle(2**10)
train = train.batch(batch_size)
train = train.prefetch(tf.data.AUTOTUNE)

del trainx, trainy

val = tf.data.Dataset.from_tensor_slices((valx, valy))
val = val.cache()
val = val.batch(batch_size)
val = val.prefetch(tf.data.AUTOTUNE)

del valx, valy

2025-09-01 16:54:53.217072: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1756756493.217115   58920 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1756756493.267680   58920 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1402 MB memory:  -> device: 0, name: NVIDIA GeForce MX350, pci bus id: 0000:01:00.0, compute capability: 6.1


In [3]:
input_ = lay.Input((32, 32, 3))
x = input_

skip = lay.Conv2D(64, (1, 1), (1, 1), "same", activation="leaky_relu")(x)
x = lay.Conv2D(64, (3, 3), (1, 1), "same", activation="leaky_relu")(x)
x = lay.Conv2D(64, (3, 3), (1, 1), "same", activation="leaky_relu")(x) + skip
x = lay.MaxPool2D()(x)

skip = lay.Conv2D(128, (1, 1), (1, 1), "same", activation="leaky_relu")(x)
x = lay.Conv2D(128, (3, 3), (1, 1), "same", activation="leaky_relu")(x)
x = lay.Conv2D(128, (3, 3), (1, 1), "same", activation="leaky_relu")(x) + skip
x = lay.MaxPool2D()(x)

skip = lay.Conv2D(256, (1, 1), (1, 1), "same", activation="leaky_relu")(x)
x = lay.Conv2D(256, (3, 3), (1, 1), "same", activation="leaky_relu")(x)
x = lay.Conv2D(256, (3, 3), (1, 1), "same", activation="leaky_relu")(x) + skip
x = lay.MaxPool2D()(x)

x = lay.Flatten()(x)
x = lay.Dense(128, "leaky_relu")(x)
x = lay.Dense(10, "softmax")(x)

model = keras.Model(input_, x)
keras.models.save_model(model, "model.keras")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["Accuracy"],
)

model.summary()

2025-09-01 16:55:10.153850: W external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:237] Falling back to the CUDA driver for PTX compilation; ptxas does not support CC 6.1
2025-09-01 16:55:10.153909: W external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:240] Used ptxas at /usr/local/cuda/bin/ptxas
2025-09-01 16:55:10.154010: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:188] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2025-09-01 16:55:10.158229: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:188] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2025-09-01 16:55:10.160608: W tensorflow/compiler/mlir/tools/kernel_gen/transforms/gpu_kernel_to_blob_pass.cc:188] Failed to compile generated PTX with ptxas. Falling back to compilation by driver.
2025-09-01 16:55:10.163054: W tensorflow/compiler/mlir/tools/kernel_gen/t

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     36,928 │ conv2d_1[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │        256 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 32,    │          0 │ conv2d_2[0][0],   │
│                     │ 64)               │            │ conv2d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 16, 16,    │          0 │ add[0][0]         │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 16, 16,    │    147,584 │ conv2d_4[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 16, 16,    │      8,320 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 16, 16,    │          0 │ conv2d_5[0][0],   │
│                     │ 128)              │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 8, 8, 128) │          0 │ add_1[0][0]       │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 8, 8, 256) │    295,168 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 8, 8, 256) │    590,080 │ conv2d_7[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 8, 8, 256) │     33,024 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 8, 8, 256) │          0 │ conv2d_8[0][0],   │
│                     │                   │            │ conv2d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 4, 4, 256) │          0 │ add_2[0][0]       │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 4096)      │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    524,416 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,712,714 (6.53 MB)

 Trainable params: 1,712,714 (6.53 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
history = model.fit(
    train,
    epochs=50,
    validation_data=val,
    callbacks=[keras.callbacks.EarlyStopping("val_loss", patience=10)],
)

Epoch 1/50


2025-09-01 16:55:31.188996: I external/local_xla/xla/service/service.cc:163] XLA service 0x74b694006aa0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-01 16:55:31.189065: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce MX350, Compute Capability 6.1
2025-09-01 16:55:31.245594: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-09-01 16:55:31.624221: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91200
2025-09-01 16:55:31.815432: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[16,64,32,32]{3,2,1,0}, u8[0]{0}) custom-call(f32[16,3,32,32]{3,2,1,0}, f32[64,3,3,3]{3,2,1,0}, f32[64]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target

UnimplementedError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "/home/thiag/anaconda3/envs/myenv/lib/python3.9/runpy.py", line 197, in _run_module_as_main

  File "/home/thiag/anaconda3/envs/myenv/lib/python3.9/runpy.py", line 87, in _run_code

  File "/home/thiag/anaconda3/envs/myenv/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/thiag/anaconda3/envs/myenv/lib/python3.9/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/thiag/anaconda3/envs/myenv/lib/python3.9/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/thiag/anaconda3/envs/myenv/lib/python3.9/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/home/thiag/anaconda3/envs/myenv/lib/python3.9/asyncio/base_events.py", line 601, in run_forever

  File "/home/thiag/anaconda3/envs/myenv/lib/python3.9/asyncio/base_events.py", line 1905, in _run_onc

In [ ]:
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Loss Curve")
plt.show()

plt.plot(history.history["Accuracy"])
plt.plot(history.history["val_Accuracy"])
plt.title("Accuracy Curve")
plt.show()